In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.health_metrics AS (
    SELECT
        b.lead_id,
        b.status_label,
        b.date_updated,
        li.lead_name,
        li.lead_display_name,
        le.customer_emails,
        b.csm_id,
        csm.csm_name,
        csm.csm_email,
        b.Contracted_Value,
        b.Tier,
        b.Avatar,
        sd.date_of_sale,
        sd.program,
        sd.setter_name,
        sd.closer_name,
        sd.contract_value AS sale_contract_value,
        e.days_since_last_email,
        m.days_since_last_meeting,
        COALESCE(mc.missed_calls_14_days, 0) AS missed_calls_14_days,
        nm.days_until_next_meeting,
        l.days_since_last_login,
        cl.days_since_last_meeting_calendly,
        s.sentiment,
        s.days_since_last_message_from_client,
        s.days_since_last_message_from_team_member
    FROM crm_ingestion.gold.customer_health_base b
    LEFT JOIN crm_ingestion.gold.lead_info li ON b.lead_id = li.lead_id
    LEFT JOIN crm_ingestion.gold.lead_emails le ON b.lead_id = le.lead_id
    LEFT JOIN crm_ingestion.gold.sales_details sd ON b.lead_id = sd.lead_id
    LEFT JOIN crm_ingestion.gold.close_crm_users_processed csm ON b.csm_id = csm.csm_id
    LEFT JOIN crm_ingestion.gold.days_since_last_email e ON b.lead_id = e.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_meeting m ON b.lead_id = m.lead_id
    LEFT JOIN crm_ingestion.gold.missed_calls_14_days mc ON b.lead_id = mc.lead_id
    LEFT JOIN crm_ingestion.gold.days_until_next_meeting nm ON b.lead_id = nm.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_login l ON b.lead_id = l.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_meeting_calendly cl ON b.lead_id = cl.lead_id
    LEFT JOIN crm_ingestion.gold.sentiment s ON b.lead_id = s.lead_id
)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_health_profile_source AS (
WITH flags AS (
    SELECT
        h.*,
        CASE
            WHEN h.days_since_last_message_from_client IS NULL THEN NULL
            WHEN h.days_since_last_message_from_client <= 14 THEN 1
            ELSE 0
        END AS slack_engaged_flag,
        CASE
            WHEN h.days_since_last_login IS NULL THEN NULL
            WHEN h.days_since_last_login <= 14 THEN 1
            ELSE 0
        END AS platform_engaged_flag,
        CASE
            WHEN h.days_since_last_meeting IS NULL OR h.days_since_last_meeting_calendly IS NULL THEN NULL
            WHEN h.days_since_last_meeting <= 30 OR h.days_since_last_meeting_calendly <= 30 THEN 1
            ELSE 0
        END AS meeting_engaged_flag
    FROM crm_ingestion.gold.health_metrics h
),

-- --- Engagement pattern + adjustments (spec 5.1 ordered rules) ---
patterns AS (
    SELECT
        f.*,
        CASE
            WHEN f.slack_engaged_flag = 1 AND f.platform_engaged_flag = 1 AND f.meeting_engaged_flag = 1 THEN 'SLACK + PLATFORM + MEETING'
            WHEN f.slack_engaged_flag = 1 AND f.platform_engaged_flag = 1 AND (f.meeting_engaged_flag = 0 OR f.meeting_engaged_flag IS NULL) THEN 'SLACK + PLATFORM'
            WHEN (f.slack_engaged_flag = 0 OR f.slack_engaged_flag IS NULL) AND f.platform_engaged_flag = 1 AND f.meeting_engaged_flag = 1 THEN 'PLATFORM + MEETING'
            WHEN f.slack_engaged_flag = 1 AND (f.platform_engaged_flag = 0 OR f.platform_engaged_flag IS NULL) AND f.meeting_engaged_flag = 1 THEN 'SLACK + MEETING'
            WHEN f.slack_engaged_flag = 1 AND (f.platform_engaged_flag = 0 OR f.platform_engaged_flag IS NULL) AND (f.meeting_engaged_flag = 0 OR f.meeting_engaged_flag IS NULL) THEN 'SLACK ONLY'
            WHEN (f.slack_engaged_flag = 0 OR f.slack_engaged_flag IS NULL) AND f.platform_engaged_flag = 1 AND (f.meeting_engaged_flag = 0 OR f.meeting_engaged_flag IS NULL) THEN 'PLATFORM ONLY'
            WHEN (f.slack_engaged_flag = 0 OR f.slack_engaged_flag IS NULL) AND (f.platform_engaged_flag = 0 OR f.platform_engaged_flag IS NULL) AND f.meeting_engaged_flag = 1 THEN 'MEETING ONLY'
            WHEN f.slack_engaged_flag = 0 AND f.platform_engaged_flag = 0 AND f.meeting_engaged_flag = 0 THEN 'NO ENGAGEMENT'
            WHEN f.slack_engaged_flag = 0 AND f.platform_engaged_flag = 0 AND f.meeting_engaged_flag IS NULL THEN 'NO ENGAGEMENT'
            WHEN f.slack_engaged_flag IS NULL AND f.platform_engaged_flag = 0 AND f.meeting_engaged_flag = 0 THEN 'NO ENGAGEMENT'
            WHEN f.slack_engaged_flag = 0 AND f.platform_engaged_flag IS NULL AND f.meeting_engaged_flag = 0 THEN 'NO ENGAGEMENT'
            ELSE 'NO DATA'
        END AS engagement_pattern,
        CASE
            WHEN f.missed_calls_14_days = 0 THEN 5
            WHEN f.missed_calls_14_days BETWEEN 1 AND 2 THEN 0
            WHEN f.missed_calls_14_days >= 3 THEN -15
        END AS missed_call_adjustment,
        CASE
            WHEN UPPER(TRIM(f.sentiment)) = 'POSITIVE' THEN 10
            WHEN UPPER(TRIM(f.sentiment)) = 'NEUTRAL' THEN 5
            WHEN UPPER(TRIM(f.sentiment)) = 'NEGATIVE' THEN -25
            ELSE 0
        END AS sentiment_adjustment
    FROM flags f
),

-- --- Base score from engagement pattern ---
scores AS (
    SELECT
        p.*,
        CASE p.engagement_pattern
            WHEN 'SLACK + PLATFORM + MEETING' THEN 90
            WHEN 'SLACK + PLATFORM' THEN 80
            WHEN 'PLATFORM + MEETING' THEN 75
            WHEN 'SLACK + MEETING' THEN 60
            WHEN 'PLATFORM ONLY' THEN 50
            WHEN 'MEETING ONLY' THEN 40
            WHEN 'SLACK ONLY' THEN 30
            WHEN 'NO ENGAGEMENT' THEN 20
            ELSE NULL
        END AS base_score
    FROM patterns p
)

-- --- FINAL OUTPUT ---
SELECT
    s.lead_id,
    s.status_label,
    s.lead_name,
    s.lead_display_name,
    s.customer_emails,
    s.csm_id AS CSM,
    s.csm_name,
    s.csm_email,
    s.Contracted_Value,
    s.Tier,
    s.Avatar,
    s.date_of_sale,
    s.program,
    s.setter_name,
    s.closer_name,
    s.sale_contract_value,
    s.days_since_last_email AS calc_days_since_last_email,
    s.days_since_last_meeting AS calc_days_since_last_meeting,
    s.missed_calls_14_days,
    s.days_until_next_meeting AS calc_days_until_next_meeting,
    s.days_since_last_login AS calc_days_since_last_login,
    s.days_since_last_meeting_calendly AS calc_days_since_last_meeting_calendly,
    COALESCE(UPPER(TRIM(s.sentiment)), 'NO_SENTIMENT') AS sentiment,
    s.days_since_last_message_from_client,
    s.days_since_last_message_from_team_member,
    s.slack_engaged_flag,
    s.platform_engaged_flag,
    s.meeting_engaged_flag,
    s.engagement_pattern,
    s.base_score,
    s.missed_call_adjustment,
    s.sentiment_adjustment,
    CASE
        WHEN s.engagement_pattern = 'NO DATA' THEN NULL
        ELSE s.base_score + s.missed_call_adjustment + s.sentiment_adjustment
    END AS final_health_score,
    CASE
        WHEN s.engagement_pattern = 'NO DATA' THEN 'NO DATA'
        WHEN s.base_score + s.missed_call_adjustment + s.sentiment_adjustment >= 75 THEN 'GOOD'
        WHEN s.base_score + s.missed_call_adjustment + s.sentiment_adjustment >= 50 THEN 'AVERAGE'
        ELSE 'POOR'
    END AS health_band
FROM scores s
)

In [0]:
%sql
-- One-time bootstrap: creates the target table ONLY if it does not exist yet
-- (e.g., first run in a fresh environment). On daily job runs this is a no-op;
-- the MERGE in the next cell handles all updates and inserts.

CREATE TABLE IF NOT EXISTS crm_ingestion.gold.customer_health_profile AS
SELECT * FROM customer_health_profile_source

In [0]:
%sql
-- SCD Type 1 upsert (current state only, no history):
--   matched lead_id  -> update all columns in place
--   new lead_id      -> insert as a new row

MERGE INTO crm_ingestion.gold.customer_health_profile AS t
USING customer_health_profile_source AS s
ON t.lead_id = s.lead_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *